### 1. Installation d'Unsloth

In [1]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
    !pip install unsloth_zoo
else:
    import torch; v = re.match(r"[0-9]{1,}\.[0-9]{1,}", str(torch.__version__)).group(0)
    xformers = "xformers==" + ("0.0.33.post1" if v=="2.9" else "0.0.32.post2" if v=="2.8" else "0.0.29.post3")
    !pip install --no-deps bitsandbytes accelerate {xformers} peft trl triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2
!pip install bitsandbytes

In [2]:
import torch
from google.colab import userdata

hf_token = userdata.get('hf_token')

gpu_props = torch.cuda.get_device_properties(0)
vram_gb = round(gpu_props.total_memory / (1024**3), 2)
bf16_ok = torch.cuda.is_bf16_supported()
print(f"🖥️ GPU détecté : {gpu_props.name} ({vram_gb} Go VRAM) | Support bfloat16 natif : {bf16_ok}")

🖥️ GPU détecté : NVIDIA L4 (22.03 Go VRAM) | Support bfloat16 natif : True


### 2. Chargement & Configuration QLoRA

In [6]:
from unsloth import FastLanguageModel, is_bfloat16_supported
import unsloth.models._utils
import unsloth.models.llama

# Neutralise la fonction de télémétrie interne d'Unsloth qui causait le TimeoutError de 120s sur Python 3.13
unsloth.models._utils.get_statistics = lambda *args, **kwargs: None
unsloth.models._utils._download_statistics = lambda *args, **kwargs: None
unsloth.models.llama.get_statistics = lambda *args, **kwargs: None

max_seq_length = 2048
dtype = torch.bfloat16 if is_bfloat16_supported() else None
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Mistral-Nemo-Instruct-2407-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    token = hf_token,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 32,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 32,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = True,
    loftq_config = None,
)

==((====))==  Unsloth 2026.11.3: Fast Mistral patching. Transformers: 4.56.2.
   \\   /|    NVIDIA L4. Num GPUs = 1. Max memory: 22.034 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.14.1+cu130. CUDA: 8.9. CUDA Toolkit: 13.0. Triton: 3.8.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Unsloth 2026.11.3 patched 40 layers with 40 QKV layers, 40 O layers and 40 MLP layers.


Unsloth: compiling each transformer block as one region (MistralDecoderLayer).


### 3. Phase 1: Continued Pre-Training sur les $7\,326$ Textes Camfranglais

In [ ]:
from datasets import load_dataset, Dataset
from trl import SFTConfig, SFTTrainer
import json

CPT_HF_REPO = "Cmr237/camfranglais-corpus-cpt"
cpt_dataset = load_dataset(CPT_HF_REPO, split="train", token=hf_token)
print(f"Corpus CPT downloaded from Hugging Face : {len(cpt_dataset)} texts.")

EOS_TOKEN = tokenizer.eos_token

def format_cpt_texts(examples):
    return {"text": [t.strip() + EOS_TOKEN for t in examples["text"]]}

cpt_dataset = cpt_dataset.map(format_cpt_texts, batched=True, num_proc=2)

model.config.use_cache = False
cpt_trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = cpt_dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    packing = True,
    args = SFTConfig(
        per_device_train_batch_size = 4,
        gradient_accumulation_steps = 2,
        warmup_ratio = 0.05,
        num_train_epochs = 3,
        learning_rate = 1e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        save_strategy = "no",
        logging_steps = 5,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "cosine",
        dataloader_num_workers = 2,
        seed = 3407,
        output_dir = "outputs_cpt",
        report_to = "none",
    ),
)

cpt_trainer_stats = cpt_trainer.train()

Corpus CPT downloaded from Hugging Face : 7326 texts.


Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/7326 [00:00<?, ? examples/s]

Unsloth: Packing train dataset (num_proc=6):   0%|          | 0/7326 [00:00<?, ? examples/s]

🦥 Unsloth: Packing enabled - training is >2x faster and uses less VRAM!


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 114 | Num Epochs = 1 | Total steps = 15
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 2 x 1) = 8
 "-____-"     Trainable parameters = 114,032,640 of 12,361,815,040 (0.92% trained)


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: packed / padding-free rows run each transformer block uncompiled; only unpacked batches use the compiled blocks.
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
5,4.333200
10,3.382200
15,3.232100


### 4. Phase 2 — Supervised Fine-Tuning

On passe maintenant aux dialogues `Cmr237/camfranglais-corpus-sft`

In [ ]:
from unsloth.chat_templates import get_chat_template, train_on_responses_only

tokenizer = get_chat_template(tokenizer, chat_template="mistral")
SYSTEM_PROMPT = "Tu réponds uniquement en camfranglais"

SFT_HF_REPO = "Cmr237/camfranglais-corpus-sft"
sft_dataset_dict = load_dataset(SFT_HF_REPO, token=hf_token)
sft_train_dataset = sft_dataset_dict["train"]
sft_test_dataset = sft_dataset_dict["test"]
print(f"SFT Dataset Downloaded : {len(sft_train_dataset)} train | {len(sft_test_dataset)} test.")

def formatted_conversations(examples):
    texts = []
    for convo in examples["conversations"]:
        full_convo = [{"role": "system", "content": SYSTEM_PROMPT}] + convo
        texts.append(tokenizer.apply_chat_template(full_convo, tokenize=False, add_generation_prompt=False))
    return {"text": texts}

sft_train_dataset = sft_train_dataset.map(formatted_conversations, batched=True)
sft_test_dataset = sft_test_dataset.map(formatted_conversations, batched=True)

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = sft_train_dataset,
    eval_dataset = sft_test_dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    packing = False,
    args = SFTConfig(
        per_device_train_batch_size = 4,
        per_device_eval_batch_size = 4,
        gradient_accumulation_steps = 2,
        warmup_ratio = 0.05,
        num_train_epochs = 3,
        learning_rate = 5e-5,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        save_strategy = "epoch",
        logging_steps = 5,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "cosine",
        dataloader_num_workers = 2,
        seed = 3407,
        output_dir = "outputs_sft",
        report_to = "none",
    ),
)

trainer = train_on_responses_only(
    trainer,
    instruction_part = "[INST]",
    response_part = "[/INST]",
)

trainer_stats = trainer.train()

README.md:   0%|          | 0.00/477 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/83.5k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/12.2k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/667 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/74 [00:00<?, ? examples/s]

SFT Dataset Downloaded : 667 train | 74 test.


... (8 outputs left out)

Map:   0%|          | 0/74 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 667 | Num Epochs = 2 | Total steps = 168
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 2 x 1) = 8
 "-____-"     Trainable parameters = 114,032,640 of 12,361,815,040 (0.92% trained)


Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.
Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
5,3.902800
10,3.320300
15,3.362800
20,3.492500
25,3.366100
30,3.091300
35,3.033900
40,3.484300
45,3.100700
50,2.862500


### 5. Évaluation sur les $74$ Dialogues de `sft_test` & Génération de `test_predictions.json`

In [11]:
import math

eval_metrics = trainer.evaluate()
eval_loss = eval_metrics.get("eval_loss", 0.0)
perplexity = math.exp(eval_loss) if eval_loss < 20 else float("inf")
print(f"Eval Loss on sft_test = {eval_loss:.4f} | Perplexity = {perplexity:.2f}")

FastLanguageModel.for_inference(model)

test_predictions = []
for item in sft_test_dataset:
    convo = item["conversations"]
    user_q = convo[0]["content"]
    human_ref = convo[1]["content"]

    msgs = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_q}
    ]
    inputs = tokenizer.apply_chat_template(
        msgs,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to("cuda")

    outputs = model.generate(
        input_ids=inputs,
        max_new_tokens=128,
        temperature=0.7,
        top_p=0.9,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )
    gen_ids = outputs[0][inputs.shape[1]:]
    model_ans = tokenizer.decode(gen_ids, skip_special_tokens=True).strip()
    test_predictions.append({
        "user": user_q,
        "human_reference": human_ref,
        "model_response": model_ans
    })

with open("test_predictions.json", "w", encoding="utf-8") as f:
    json.dump({
        "eval_loss": eval_loss,
        "perplexity": perplexity,
        "predictions": test_predictions
    }, f, indent=2, ensure_ascii=False)

print(f"{len(test_predictions)} generated answers saved in test_predictions.json !")

Filter:   0%|          | 0/74 [00:00<?, ? examples/s]

Unsloth: Not an error, but `num_items_in_batch` could not be counted for MistralForCausalLM's labels, so each micro-batch loss is a mean.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Eval Loss on sft_test = 3.3905 | Perplexity = 29.68
74 generated answers saved in test_predictions.json !


### 6. Inference

In [12]:
FastLanguageModel.for_inference(model)
from transformers import TextStreamer

messages = [{"role": "system", "content": SYSTEM_PROMPT}]
while True:
    user_input = str(input("user: \n"))
    if user_input.lower() == "exit":
        break
    messages.append({"role": "user", "content": user_input})
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize = True,
        add_generation_prompt = True,
        return_tensors = "pt",
    ).to("cuda")

    text_streamer = TextStreamer(tokenizer, skip_prompt = True, skip_special_tokens = True)
    print("Assistant: ", end="", flush=True)
    outputs = model.generate(
        input_ids = inputs,
        streamer = text_streamer,
        max_new_tokens = 128,
        pad_token_id = tokenizer.eos_token_id
    )
    prompt_length = inputs.shape[1]
    response_ids = outputs[0][prompt_length:]
    response_text = tokenizer.decode(response_ids, skip_special_tokens=True).strip()
    messages.append({"role": "assistant", "content": response_text})

user: 
Yo on dit quoi ?
Assistant: On go au bar ce soir, ça fait longtemps qu'on est pas sorti.
user: 
non j'ai autres choses à do
Assistant: Tu vas travailler tous les jours, il faut aussi se détendre un peu.
user: 
autre chose que le bar bro
Assistant: On peut aller au ciné ou faire un tour au parc.
user: 
tu as déjà vu le parc où au camer bro  ?
Assistant: Tu as déjà vu le parc où au Camer ?
user: 
il n'y en a pas
Assistant: Il y'a le parc de la joie à Yaoundé et le parc des Princes à Douala.
user: 
aka laisse ça 
Assistant: C'est toi qui as commencé avec le parc.
user: 
et il y a les gos là bas ? 
Assistant: Il y'a les gos partout au Camer.
user: 
en vrai 
Assistant: Tu viens avec moi ce soir, je vais te montrer.
user: 
euil
Assistant: Je te dis.
user: 
donc tu es high hein
Assistant: Je suis juste inspiré par l'ambiance.


KeyboardInterrupt: Interrupted by user

### 7. Sauvegarde des Adaptateurs LoRA & Export GGUF (`q4_k_m`) vers Hugging Face

In [ ]:
model.push_to_hub("Cmr237/Camfranglais", token = hf_token)
tokenizer.push_to_hub("Cmr237/Camfranglais", token = hf_token)

model.push_to_hub_gguf(
    "Cmr237/Camfranglais",
    tokenizer,
    quantization_method = "q4_k_m",
    token = hf_token,
)